<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Structured Output
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Das Pydantic-Schema ist der **Agenten-Vertrag** zwischen dem Meeting- & Research-Briefing-Agent und den nachgelagerten Schritten (Freigabe, Anzeige, Weiterverarbeitung) — die Antwort muss Quelle, Sicherheit und Hinweis strukturiert enthalten, sonst lässt sie sich nicht kontrolliert prüfen. Structured Output ist damit Antwort-/Evidenzschema und technische Grundlage für **Prüfen**.

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M04-Structured-Output"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

# 1 | Übersicht
---


Strukturierte Ausgaben sind keine Komfortfunktion — sie sind die Voraussetzung dafür, dass LLM-Ausgaben zuverlässig weiterverarbeitet werden können.

| Problem | Lösung |
|---|---|
| LLM gibt Freitext → schwer weiterverarbeitbar | `with_structured_output()` + Pydantic-Schema |
| Regex-Parsing → fehleranfällig | Automatische Validierung durch Pydantic |
| Kein Type-Checking | Typsichere Python-Objekte |


In [ ]:
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    INPUT["Freier Text\n'RAGAS bewertet RAG-Systeme'"]
    SCHEMA["Pydantic Schema\nPaperNotiz(titel, methode, nutzen)"]
    LLM["LLM\ngpt-5.4-mini"]
    OUTPUT["Python-Objekt\nPaperNotiz(... )"]

    INPUT --> LLM
    SCHEMA --> LLM
    LLM --> OUTPUT
'''

mermaid(diagram, width=750)

# 2 | Pydantic Basics
---



Pydantic ist die Grundlage für strukturierte Ausgaben. Ein **Pydantic-Modell** definiert:

- **Felder** mit Typ-Annotationen (`str`, `int`, `List[str]`, …)
- **Beschreibungen** per `Field(description=...)` – das LLM liest diese!
- **Optionale Felder** mit `Optional[...]` und `default=None`
- **Validierungslogik** automatisch durch Pydantic

<p><font color='darkblue' size="4">💡 <b>Merkregel</b></font></p>

Der `description`-Text in `Field()` ist der Prompt-Hinweis für das LLM. Je klarer die Beschreibung, desto besser das Ergebnis.

In [ ]:
import json
from typing import Optional, List, Literal
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# Konfigurationskonstanten
MAX_RETRIES = 3   # API-Retry-Versuche bei kurzzeitigen Störungen

llm = init_chat_model(BASELINE)

# Einfaches Schema
class PaperKurznotiz(BaseModel):
    """Strukturierte Kurznotiz zu einem Paper oder Studienauszug."""
    titel: str = Field(description="Titel oder Kurzname des Papers")
    jahr: Optional[int] = Field(default=None, description="Publikationsjahr, falls genannt")
    methode: str = Field(description="Zentrale Methode oder Evaluationsidee")

# Schema inspizieren - was das LLM sieht
mprint("## Pydantic-Schema")
mprint(f"**Felder:** `{list(PaperKurznotiz.model_fields.keys())}`")

schema_json = json.dumps(PaperKurznotiz.model_json_schema(), indent=2, ensure_ascii=False)
mprint(f"**JSON Schema:**\n```json\n{schema_json}\n```")
print(f"   MAX_RETRIES = {MAX_RETRIES}")


# 3 | with_structured_output()
---



`with_structured_output()` bindet ein Pydantic-Schema direkt ans LLM:

```python
structured_llm = llm.with_structured_output(MeinSchema)
ergebnis = structured_llm.invoke("...")
**ergebnis ist ein MeinSchema-Objekt – typsicher!**
```

**Was intern passiert:**
1. LangChain schickt das JSON Schema des Pydantic-Modells an das LLM
2. Das LLM füllt die Felder anhand der `description`-Texte
3. LangChain validiert die Antwort und gibt ein typsicheres Objekt zurück

In [ ]:
class PaperKurznotiz(BaseModel):
    """Strukturierte Kurznotiz zu einem Paper oder Studienauszug."""
    titel: str = Field(description="Titel oder Kurzname des Papers")
    jahr: Optional[int] = Field(default=None, description="Publikationsjahr, falls genannt")
    methode: str = Field(description="Zentrale Methode oder Evaluationsidee")

# LLM mit Schema binden + with_retry gegen transiente API-Fehler
structured_llm = llm.with_structured_output(PaperKurznotiz).with_retry(stop_after_attempt=MAX_RETRIES)
paper_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_studien_zusammenfassung_prompt.md", mode="T")

# Aufruf - LLM gibt ein PaperKurznotiz-Objekt zurück
ergebnis = (paper_prompt | structured_llm).invoke(
    {"text": "RAGAS beschreibt Metriken, um RAG-Systeme entlang von Faithfulness, Answer Relevancy und Kontextqualität zu bewerten."}
)

mprint("## Ergebnis")
mprint(f"**Typ:** `{type(ergebnis).__name__}`")
mprint(f"**Titel:** {ergebnis.titel}")
mprint(f"**Jahr:** {ergebnis.jahr}")
mprint(f"**Methode:** {ergebnis.methode}")

**Was passiert hier?**

1. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte JSON-Ausgabe

# 4 | Praktische Anwendungen
---



Zwei häufige Anwendungsfälle:

**Informationsextraktion** – strukturierte Daten aus Freitext extrahieren

**Klassifikation** – Text in vordefinierte Kategorien einordnen (mit `Literal[...]`)

In [ ]:
class ResearchSignal(BaseModel):
    """Klassifiziertes Briefing-Signal."""
    signaltyp: Literal["methode", "evaluation", "risiko", "anwendung"] = Field(
        description="Kategorie des Signals im Briefing-Agent-Kontext"
    )
    relevanz: Literal["niedrig", "mittel", "hoch"] = Field(
        description="Relevanz für den Meeting-Briefing-Agent"
    )
    zusammenfassung: str = Field(description="Kurze Zusammenfassung des Signals in einem Satz")

classifier = llm.with_structured_output(ResearchSignal).with_retry(stop_after_attempt=MAX_RETRIES)
signal_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_research_signal_classification_prompt.md", mode="T")

signale = [
    "RAGAS bewertet Faithfulness und Answer Relevancy für RAG-Antworten.",
    "Irrelevante Retrieval-Kontexte können die Antwortqualität verschlechtern.",
    "Ein Quellen-Gate verhindert Antworten ohne belegten Kontext.",
]

mprint("## Klassifizierte Briefing-Signale")
for text in signale:
    signal = (signal_prompt | classifier).invoke({"text": text})
    mprint("")
    mprint(f"**Signal:** {text}")
    mprint(f"-> `{signal.signaltyp}` | Relevanz: `{signal.relevanz}`")
    mprint(f"-> {signal.zusammenfassung}")

# 5 | Structured Output in der Praxis
---



Structured Output eignet sich überall dort, wo das LLM-Ergebnis direkt weiterverarbeitet werden soll:
- Formular-Ausfüllung aus Freitext
- Analyse-Pipelines mit definierten Ausgabeformaten
- Bewertung und Scoring von Texten

Der Unterschied zu freiem Text: Das Ergebnis ist **sofort als Python-Objekt nutzbar** – kein Parsing, keine Fehlerbehandlung für Format-Abweichungen.

In [ ]:
class Quellenkarte(BaseModel):
    """Strukturierter Quellenhinweis für eine Research-Antwort."""
    quelle: str = Field(description="Dokument, Paper oder Kurzkennung der Quelle")
    aussage: str = Field(description="Belegte Kernaussage aus der Quelle")
    citation: str = Field(description="Kurzer Citation-Hinweis für die spätere Antwort")
    abschnitt: Optional[str] = Field(default=None, description="Abschnitt oder Seitenhinweis, falls genannt")

quellen_llm = llm.with_structured_output(Quellenkarte).with_retry(stop_after_attempt=MAX_RETRIES)
quellen_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_citation_format_prompt.md", mode="T")

text = """
Quelle: ragas_evaluation.pdf, Abschnitt Evaluation.
RAGAS trennt die Bewertung von Antworttreue und Antwortrelevanz und eignet sich deshalb
als Baustein für Regressionstests in RAG-Systemen.
"""

q = (quellen_prompt | quellen_llm).invoke({"text": text})

mprint("## Quellenkarte")
mprint(f"**Quelle:** {q.quelle}")
mprint(f"**Aussage:** {q.aussage}")
mprint(f"**Citation:** {q.citation}")
if q.abschnitt:
    mprint(f"**Abschnitt:** {q.abschnitt}")

# 6 | LangSmith: Structured Output Traces
---



LangSmith macht Structured-Output-Aufrufe vollständig sichtbar:
- Das gesendete **JSON Schema** (Pydantic-Modell)
- Die **LLM-Antwort** vor der Validierung
- Das **valide Python-Objekt** nach der Validierung


In [ ]:
# 6.1 Structured Output Trace in LangSmith
class StudienReview(BaseModel):
    """Strukturierte Bewertung eines Paper-Auszuges."""
    titel: str = Field(description="Titel oder Kurzname des Papers")
    evidenz_score: int = Field(description="Evidenzscore von 1 (schwach) bis 5 (stark)")
    nutzen: List[str] = Field(description="Nutzenaspekte für den Meeting-Briefing-Agent")
    grenzen: List[str] = Field(description="Grenzen oder Risiken des Ansatzes")

# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M04_Kap6_StructuredTrace",
    "tags": ["m04", "structured-output", "langsmith"]
}

# 2. with_structured_output() + with_retry() + with_config()
review_llm = (
    llm
    .with_structured_output(StudienReview)
    .with_retry(stop_after_attempt=MAX_RETRIES)
    .with_config(**run_cfg)
)
review_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_research_review_prompt.md", mode="T")

auszug = """
RAGAS ist ein Framework zur Evaluation von RAG-Systemen. Es bewertet unter anderem
Faithfulness und Answer Relevancy und kann dadurch Regressionen in Antworten sichtbar
machen. Grenzen entstehen, wenn Referenzdaten oder LLM-Judge-Konfigurationen nicht stabil sind.
"""
ergebnis = (review_prompt | review_llm).invoke({"text": auszug})

mprint("## Trace erstellt")
mprint(f"**Titel:** {ergebnis.titel}")
mprint(f"**Evidenz:** {ergebnis.evidenz_score}/5")
mprint(f"**Nutzen:** {', '.join(ergebnis.nutzen)}")
mprint(f"**Grenzen:** {', '.join(ergebnis.grenzen)}")

**Was passiert hier?**

1. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte JSON-Ausgabe

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M04-Structured-Output", limit=3, show_steps=True)

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> Generative KI kann als Unterstützung beim Lernen und Entwickeln genutzt werden. Bei Fehlermeldungen, Teilproblemen oder Code-Varianten kann zum Beispiel Gemini in Google Colab helfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Strukturierte Ausgaben für Paper-Signale
</font></p>

Der Meeting-Briefing-Agent muss freie Textpassagen in belastbare Daten überführen: Paper-Titel, Methode, Evidenzstärke, Risiken und offene Fragen. M04 übt diese Transformation mit Pydantic und `with_structured_output()`.


**Grundlagen**
- Ein Pydantic-Modell mit mindestens drei Feldern für eine Paper-Notiz definieren.
- Einen strukturierten LLM mit `with_structured_output()` erstellen.
- Eine Beispielpassage extrahieren.
- Modell in `mein_schema`, strukturierten LLM in `mein_llm` speichern.

**✅ Erledigt wenn:** Die Rückgabe ist eine Instanz von `mein_schema`, kein Freitext und kein Dictionary.


In [ ]:
# Grundlagen: Pydantic-Schema + with_structured_output
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class PaperNotiz(BaseModel):
    titel: str = Field(description="Titel oder Kurzname des Papers")
    methode: str = Field(description="Zentrale Methode oder Technik")
    nutzen: str = Field(description="Wichtigster Nutzen für den Meeting-Briefing-Agent")

# ...


**Aufbau**
- Das Schema auf mindestens vier Felder erweitern.
- Mindestens ein optionales Feld ergänzen.
- Drei unterschiedliche Beispieltexte testen.
- Ergebnisse in `test_ergebnisse` speichern.

**✅ Erledigt wenn:** Alle Test-Ergebnisse sind valide Pydantic-Objekte und mindestens ein Feld ist optional.


In [ ]:
# Aufbau: erweitertes Schema mit optionalem Feld und drei Testfällen
from typing import Optional


class PaperSignal(BaseModel):
    titel: str = Field(description="Titel oder Kurzname des Papers")
    kernidee: str = Field(description="Kernaussage in einem kurzen Satz")
    methode: str = Field(description="Methode, Framework oder Evaluationsansatz")
    relevanz: str = Field(description="Relevanz für den Meeting-Briefing-Agent")
    risiko: Optional[str] = Field(default=None, description="Genanntes Risiko oder offene Grenze")

# ...

**Vertiefung**
- Ein `Literal[...]`-Feld für eine Klassifikation ergänzen.
- Ein Nested Schema für Quellenhinweise verwenden.
- Extraktion und Klassifikation in einem kombinierten Schema testen.
- Kombiniertes Schema in `mein_kombinations_schema` speichern.

**✅ Erledigt wenn:** Das kombinierte Schema enthält ein Literal-Feld, ein eingebettetes Sub-Modell und erzeugt valide Ergebnisse.


In [ ]:
# Vertiefung: Literal + Nested Schema
from typing import Literal


class QuellenHinweis(BaseModel):
    dokument_id: str = Field(description="Kurzkennung des Projekt- oder Fachartikeldokuments")
    abschnitt: Optional[str] = Field(default=None, description="Abschnitt oder Seitenhinweis, falls bekannt")


class BewertetesBriefingSignal(BaseModel):
    titel: str = Field(description="Titel oder Kurzname des Dokuments")
    signaltyp: Literal["entscheidung", "risiko", "offene_frage", "evidenz"] = Field(
        description="Art des extrahierten Signals"
    )
    zusammenfassung: str = Field(description="Kompakte fachliche Zusammenfassung")
    quelle: QuellenHinweis = Field(description="Quellenhinweis zum Signal")


# ...

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)
- [KI-Agent](https://editor.p5js.org/ralf.bendig.rb/full/u3Ee0jtFo)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [LangChain Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langchain-best-practices.html)
- [Prompt Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/prompt-engineering.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
- [Evaluation & Observability](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/evaluation-observability.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
